In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark.conf.set(
    "spark.sql.session.timeZone",
    "UTC"
)

raw_root = "s3://l1l2bucket/data/raw/coindcx"

catalog = "crypto_lakehouse"
bronze_schema = "bronze"
silver_schema = "silver"
gold_schema = "gold"

display(
    dbutils.fs.ls(raw_root)
)

path,name,size,modificationTime
s3://l1l2bucket/data/raw/coindcx/balances/,balances/,0,1790360782562
s3://l1l2bucket/data/raw/coindcx/crypto_deposits/,crypto_deposits/,0,1790360782562
s3://l1l2bucket/data/raw/coindcx/crypto_withdrawals/,crypto_withdrawals/,0,1790360782562
s3://l1l2bucket/data/raw/coindcx/fiat_deposits/,fiat_deposits/,0,1790360782562
s3://l1l2bucket/data/raw/coindcx/fiat_withdrawals/,fiat_withdrawals/,0,1790360782562
s3://l1l2bucket/data/raw/coindcx/ingestion_manifest/,ingestion_manifest/,0,1790360782562
s3://l1l2bucket/data/raw/coindcx/instant_orders/,instant_orders/,0,1790360782562


####CoinDcx balances

In [0]:
balances_schema = StructType([
    StructField("balance", DecimalType(38, 18), True),
    StructField("currency", StringType(), True),
    StructField("locked_balance", DecimalType(38, 18), True)
])

In [0]:
balances_raw_df = (
    spark.read
    .schema(balances_schema)
    .option("multiline", True)
    .option("recursiveFileLookup", True)
    .json(f"{raw_root}/balances/")
)

balances_bronze_df = (
    balances_raw_df
    .select(
        "*",
        col("_metadata.file_name").alias("source_file_name"),
        col("_metadata.file_path").alias("_source_file")
    )
    .withColumn("exchange", lit("CoinDCX"))
    .withColumn("source_type", lit("Balance"))
    .withColumn("ingested_at", current_timestamp())
    .dropDuplicates([
        "currency",
        "_source_file"
    ])
)

print("balance rows:", balances_bronze_df.count())

display(
    balances_bronze_df.orderBy(
        col("currency").desc()
    )
)

balance rows: 4


balance,currency,locked_balance,source_file_name,_source_file,exchange,source_type,ingested_at
0E-18,USDT,0E-18,balances_20260925T181928_277012Z_c83f7988.json,s3://l1l2bucket/data/raw/coindcx/balances/balances_20260925T181928_277012Z_c83f7988.json,CoinDCX,Balance,2026-09-25T18:26:37.291Z
0.121948825900600000,SOL,0E-18,balances_20260925T181928_277012Z_c83f7988.json,s3://l1l2bucket/data/raw/coindcx/balances/balances_20260925T181928_277012Z_c83f7988.json,CoinDCX,Balance,2026-09-25T18:26:37.291Z
0.034810000000000000,INR,0E-18,balances_20260925T181928_277012Z_c83f7988.json,s3://l1l2bucket/data/raw/coindcx/balances/balances_20260925T181928_277012Z_c83f7988.json,CoinDCX,Balance,2026-09-25T18:26:37.291Z
0E-18,BTC,0E-18,balances_20260925T181928_277012Z_c83f7988.json,s3://l1l2bucket/data/raw/coindcx/balances/balances_20260925T181928_277012Z_c83f7988.json,CoinDCX,Balance,2026-09-25T18:26:37.291Z


In [0]:
(
    balances_bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        True
    )
    .saveAsTable(
        f"{catalog}.{bronze_schema}.coindcx_balances"
    )
)


print(
    "Saved CoinDCX balance rows:",
    spark.table(
        f"{catalog}.{bronze_schema}.coindcx_balances"
    ).count()
)

Saved CoinDCX balance rows: 4


####CoinDCX instant orders

In [0]:
instant_orders_schema = StructType([
    StructField("transactionId", StringType(), True),
    StructField("tradeSource", StringType(), True),
    StructField("asset", StringType(), True),
    StructField("quoteAsset", StringType(), True),
    StructField("side", StringType(), True),
    StructField("orderType", StringType(), True),
    StructField("status", StringType(), True),
    StructField("quantity", StringType(), True),
    StructField("averagePriceInr", StringType(), True),
    StructField("grossAmountInr", StringType(), True),
    StructField("feeInr", StringType(), True),
    StructField("netAmountInr", StringType(), True),
    StructField("tdsInr", StringType(), True),
    StructField("transactionTime", StringType(), True)
])

instant_orders_raw_df = (
    spark.read
    .schema(instant_orders_schema)
    .option("multiline", True)
    .option("recursiveFileLookup", True)
    .json(f"{raw_root}/instant_orders/")
)

instant_orders_bronze_df = (
    instant_orders_raw_df
    .select(
        "*",
        col("_metadata.file_name").alias("source_file_name"),
        col("_metadata.file_path").alias("_source_file")
    )
    .withColumn("exchange", lit("CoinDCX"))
    .withColumn("source_type", lit("Instant Order"))
    .withColumn("ingested_at", current_timestamp())
    .dropDuplicates([
        "transactionId",
    ])
)

print(
    "CoinDCX Bronze Instant orders:",
    instant_orders_bronze_df.count()
)

display(instant_orders_bronze_df)

CoinDCX Bronze Instant orders: 1


transactionId,tradeSource,asset,quoteAsset,side,orderType,status,quantity,averagePriceInr,grossAmountInr,feeInr,netAmountInr,tdsInr,transactionTime,source_file_name,_source_file,exchange,source_type,ingested_at
292a7605-4955-4d01-b296-41d32c14920e,Instant,SOL,INR,BUY,INSTANT,COMPLETED,0.1219488259006,8151.78,994.10,5.86519,999.96519,null,2026-03-14T21:20:21+05:30,instant_orders_20260925T181928_600854Z_a4900abc.json,s3://l1l2bucket/data/raw/coindcx/instant_orders/instant_orders_20260925T181928_600854Z_a4900abc.json,CoinDCX,Instant Order,2026-09-25T18:26:48.365Z


In [0]:
(
    instant_orders_bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        True
    )
    .saveAsTable(
        f"{catalog}.{bronze_schema}.coindcx_instant_orders"
    )
)


print(
    "Saved CoinDCX Instant orders:",
    spark.table(
        f"{catalog}.{bronze_schema}.coindcx_instant_orders"
    ).count()
)

Saved CoinDCX Instant orders: 1


####CoinDCX Fiat deposits

In [0]:
fiat_deposits_schema = StructType([
    StructField("transactionId", StringType(), True),
    StructField("asset", StringType(), True),
    StructField("amount", StringType(), True),
    StructField("status", StringType(), True),
    StructField("paymentMethod", StringType(), True),
    StructField("createdAt", StringType(), True),
    StructField("updatedAt", StringType(), True)
])


fiat_deposits_raw_df = (
    spark.read
    .schema(fiat_deposits_schema)
    .option("multiline", True)
    .option("recursiveFileLookup", True)
    .json(f"{raw_root}/fiat_deposits/")
)

fiat_deposits_bronze_df = (
    fiat_deposits_raw_df
    .select(
        "*",
        col("_metadata.file_name").alias("source_file_name"),
        col("_metadata.file_path").alias("_source_file")
    )
    .withColumn("exchange", lit("CoinDCX"))
    .withColumn("source_type", lit("Fiat Deposit"))
    .withColumn("ingested_at", current_timestamp())
    .dropDuplicates([
        "transactionId",
    ])
)

print(
    "CoinDCX Bronze fiat deposit rows:",
    fiat_deposits_bronze_df.count()
)

display(fiat_deposits_bronze_df)

CoinDCX Bronze fiat deposit rows: 1


transactionId,asset,amount,status,paymentMethod,createdAt,updatedAt,source_file_name,_source_file,exchange,source_type,ingested_at
422144727195,INR,1000.00,COMPLETED,Pay via Instant UPI,2026-03-14T21:06:00+05:30,2026-03-14T21:06:00+05:30,fiat_deposits_20260925T181928_745720Z_6855d341.json,s3://l1l2bucket/data/raw/coindcx/fiat_deposits/fiat_deposits_20260925T181928_745720Z_6855d341.json,CoinDCX,Fiat Deposit,2026-09-25T18:26:59.260Z


In [0]:
(
    fiat_deposits_bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        True
    )
    .saveAsTable(
        f"{catalog}.{bronze_schema}.coindcx_fiat_deposits"
    )
)


print(
    "Saved CoinDCX fiat deposits:",
    spark.table(
        f"{catalog}.{bronze_schema}.coindcx_fiat_deposits"
    ).count()
)

Saved CoinDCX fiat deposits: 1


####CoinDCX fiat withdrawals

In [0]:
fiat_withdrawals_schema = StructType([
    StructField("transactionId", StringType(), True),
    StructField("asset", StringType(), True),
    StructField("amount", StringType(), True),
    StructField("status", StringType(), True),
    StructField("paymentMethod", StringType(), True),
    StructField("createdAt", StringType(), True),
    StructField("updatedAt", StringType(), True)
])


fiat_withdrawals_raw_df = (
    spark.read
    .schema(fiat_withdrawals_schema)
    .option("multiline", True)
    .option("recursiveFileLookup", True)
    .json(f"{raw_root}/fiat_withdrawals/")
)


fiat_withdrawals_bronze_df = (
    fiat_withdrawals_raw_df
    .select(
        "*",

        col("_metadata.file_name")
        .alias("source_file_name"),

        col("_metadata.file_path")
        .alias("_source_file")
    )
    .withColumn(
        "exchange",
        lit("CoinDCX")
    )
    .withColumn(
        "source_type",
        lit("Fiat Withdrawal")
    )
    .withColumn(
        "ingested_at",
        current_timestamp()
    )
    .dropDuplicates([
        "transactionId"
    ])
)


print(
    "CoinDCX fiat withdrawal rows:",
    fiat_withdrawals_bronze_df.count()
)

fiat_withdrawals_bronze_df.printSchema()

CoinDCX fiat withdrawal rows: 0
root
 |-- transactionId: string (nullable = true)
 |-- asset: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- status: string (nullable = true)
 |-- paymentMethod: string (nullable = true)
 |-- createdAt: string (nullable = true)
 |-- updatedAt: string (nullable = true)
 |-- source_file_name: string (nullable = false)
 |-- _source_file: string (nullable = false)
 |-- exchange: string (nullable = false)
 |-- source_type: string (nullable = false)
 |-- ingested_at: timestamp (nullable = false)



In [0]:
(
    fiat_withdrawals_bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        True
    )
    .saveAsTable(
        f"{catalog}.{bronze_schema}.coindcx_fiat_withdrawals"
    )
)


print(
    "Saved CoinDCX fiat withdrawals:",
    spark.table(
        f"{catalog}.{bronze_schema}.coindcx_fiat_withdrawals"
    ).count()
)

Saved CoinDCX fiat withdrawals: 0


####CoinDCX crypto deposits 

In [0]:
crypto_deposits_schema = StructType([
    StructField("transactionId", StringType(), True),
    StructField("asset", StringType(), True),
    StructField("amount", StringType(), True),
    StructField("status", StringType(), True),
    StructField("network", StringType(), True),
    StructField("transactionHash", StringType(), True),
    StructField("fromAddress", StringType(), True),
    StructField("toAddress", StringType(), True),
    StructField("fee", StringType(), True),
    StructField("createdAt", StringType(), True),
    StructField("updatedAt", StringType(), True)
])


crypto_deposits_raw_df = (
    spark.read
    .schema(crypto_deposits_schema)
    .option("multiline", True)
    .option("recursiveFileLookup", True)
    .json(f"{raw_root}/crypto_deposits/")
)


crypto_deposits_bronze_df = (
    crypto_deposits_raw_df
    .select(
        "*",

        col("_metadata.file_name")
        .alias("source_file_name"),

        col("_metadata.file_path")
        .alias("_source_file")
    )
    .withColumn(
        "exchange",
        lit("CoinDCX")
    )
    .withColumn(
        "source_type",
        lit("Crypto Deposit")
    )
    .withColumn(
        "ingested_at",
        current_timestamp()
    )
    .dropDuplicates([
        "transactionId"
    ])
)


print(
    "CoinDCX crypto deposit rows:",
    crypto_deposits_bronze_df.count()
)

crypto_deposits_bronze_df.printSchema()

CoinDCX crypto deposit rows: 0
root
 |-- transactionId: string (nullable = true)
 |-- asset: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- status: string (nullable = true)
 |-- network: string (nullable = true)
 |-- transactionHash: string (nullable = true)
 |-- fromAddress: string (nullable = true)
 |-- toAddress: string (nullable = true)
 |-- fee: string (nullable = true)
 |-- createdAt: string (nullable = true)
 |-- updatedAt: string (nullable = true)
 |-- source_file_name: string (nullable = false)
 |-- _source_file: string (nullable = false)
 |-- exchange: string (nullable = false)
 |-- source_type: string (nullable = false)
 |-- ingested_at: timestamp (nullable = false)



In [0]:
(
    crypto_deposits_bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        True
    )
    .saveAsTable(
        f"{catalog}.{bronze_schema}.coindcx_crypto_deposits"
    )
)


print(
    "Saved CoinDCX crypto deposits:",
    spark.table(
        f"{catalog}.{bronze_schema}.coindcx_crypto_deposits"
    ).count()
)

Saved CoinDCX crypto deposits: 0


####CoinDCX crypto deposits

In [0]:
crypto_withdrawals_schema = StructType([
    StructField("transactionId", StringType(), True),
    StructField("asset", StringType(), True),
    StructField("amount", StringType(), True),
    StructField("status", StringType(), True),
    StructField("network", StringType(), True),
    StructField("transactionHash", StringType(), True),
    StructField("fromAddress", StringType(), True),
    StructField("toAddress", StringType(), True),
    StructField("fee", StringType(), True),
    StructField("createdAt", StringType(), True),
    StructField("updatedAt", StringType(), True)
])


crypto_withdrawals_raw_df = (
    spark.read
    .schema(crypto_withdrawals_schema)
    .option("multiline", True)
    .option("recursiveFileLookup", True)
    .json(f"{raw_root}/crypto_withdrawals/")
)


crypto_withdrawals_bronze_df = (
    crypto_withdrawals_raw_df
    .select(
        "*",

        col("_metadata.file_name")
        .alias("source_file_name"),

        col("_metadata.file_path")
        .alias("_source_file")
    )
    .withColumn(
        "exchange",
        lit("CoinDCX")
    )
    .withColumn(
        "source_type",
        lit("Crypto Withdrawal")
    )
    .withColumn(
        "ingested_at",
        current_timestamp()
    )
    .dropDuplicates([
        "transactionId"
    ])
)


print(
    "CoinDCX crypto withdrawal rows:",
    crypto_withdrawals_bronze_df.count()
)

crypto_withdrawals_bronze_df.printSchema()

CoinDCX crypto withdrawal rows: 0
root
 |-- transactionId: string (nullable = true)
 |-- asset: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- status: string (nullable = true)
 |-- network: string (nullable = true)
 |-- transactionHash: string (nullable = true)
 |-- fromAddress: string (nullable = true)
 |-- toAddress: string (nullable = true)
 |-- fee: string (nullable = true)
 |-- createdAt: string (nullable = true)
 |-- updatedAt: string (nullable = true)
 |-- source_file_name: string (nullable = false)
 |-- _source_file: string (nullable = false)
 |-- exchange: string (nullable = false)
 |-- source_type: string (nullable = false)
 |-- ingested_at: timestamp (nullable = false)



In [0]:
(
    crypto_withdrawals_bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        True
    )
    .saveAsTable(
        f"{catalog}.{bronze_schema}.coindcx_crypto_withdrawals"
    )
)


print(
    "Saved CoinDCX crypto withdrawals:",
    spark.table(
        f"{catalog}.{bronze_schema}.coindcx_crypto_withdrawals"
    ).count()
)

Saved CoinDCX crypto withdrawals: 0


In [0]:
coindcx_bronze_tables = [
    "coindcx_balances",
    "coindcx_instant_orders",
    "coindcx_fiat_deposits",
    "coindcx_fiat_withdrawals",
    "coindcx_crypto_deposits",
    "coindcx_crypto_withdrawals"
]


for table_name in coindcx_bronze_tables:

    row_count = spark.table(
        f"{catalog}.{bronze_schema}.{table_name}"
    ).count()

    print(
        f"{table_name}: {row_count} rows"
    )

coindcx_balances: 4 rows
coindcx_instant_orders: 1 rows
coindcx_fiat_deposits: 1 rows
coindcx_fiat_withdrawals: 0 rows
coindcx_crypto_deposits: 0 rows
coindcx_crypto_withdrawals: 0 rows
